In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ===============================
# 🔗 MOUNT GOOGLE DRIVE
# ===============================
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/major project code implementation/"

# ===============================
# INSTALL
# ===============================
!pip install xgboost tensorflow

# ===============================
# IMPORTS
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    precision_score, recall_score,
    roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
)

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from collections import Counter
import joblib

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv(SAVE_PATH + "dataset_30000.csv")
df.columns = df.columns.str.strip().str.replace(" ", "_")
df = df.drop_duplicates()

print("Dataset Shape:", df.shape)

# ===============================
# SPLIT
# ===============================
X = df.drop("ASD_traits", axis=1)
y = LabelEncoder().fit_transform(df["ASD_traits"])

# REMOVE COLUMNS
cols_remove = [
    "A10_Autism_Spectrum_Quotient",
    "Qchat_10_Score",
    "Social_Responsiveness_Scale",
    "Childhood_Autism_Rating_Scale"
]
X = X.drop(columns=cols_remove, errors='ignore')

if "CASE_NO_PATIENT'S" in X.columns:
    X = X.drop("CASE_NO_PATIENT'S", axis=1)

# FIX EMPTY
for col in X.columns:
    if X[col].isnull().all():
        X[col] = 0

# ===============================
# TRAIN TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===============================
# ENCODING
# ===============================
encoders = {}
for col in X_train.columns:
    if X_train[col].dtype == "object":
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = le.transform(X_test[col])
        encoders[col] = le

joblib.dump(encoders, SAVE_PATH + "encoders.pkl")

# Store feature names before imputation/scaling, as these operations convert to numpy arrays
original_feature_names = X_train.columns.tolist()

# ===============================
# PREPROCESSING
# ===============================
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

selector = SelectKBest(score_func=f_classif, k=20)
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

# Save the names of the selected features
selected_feature_names = selector.get_feature_names_out(input_features=original_feature_names)
joblib.dump(selected_feature_names, SAVE_PATH + "feature_names.pkl")

# ===============================
# CLASS BALANCE
# ===============================
counter = Counter(y_train)
scale_pos_weight = counter[1] / counter[0]

# ===============================
# MODELS
# ===============================
# XGBoost
xgb = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_train, y_train)

# ANN
ann = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = ann.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# ===============================
# PREDICTIONS
# ===============================
xgb_probs = xgb.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]
ann_probs = ann.predict(X_test).flatten()

xgb_pred = (xgb_probs > 0.5).astype(int)
rf_pred = (rf_probs > 0.5).astype(int)
ann_pred = (ann_probs > 0.5).astype(int)

# ===============================
# METRICS
# ===============================
metrics_df = pd.DataFrame({
    "Model": ["XGBoost", "Random Forest", "ANN"],
    "Accuracy": [
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, ann_pred)
    ],
    "Precision": [
        precision_score(y_test, xgb_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, ann_pred)
    ],
    "Recall": [
        recall_score(y_test, xgb_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, ann_pred)
    ],
    "F1 Score": [
        f1_score(y_test, xgb_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, ann_pred)
    ]
})

print(metrics_df)

metrics_df.to_csv(SAVE_PATH + "model_metrics.csv", index=False)

# ===============================
# 📈 ROC CURVE
# ===============================
plt.figure()

for name, probs in zip(
    ["XGBoost", "Random Forest", "ANN"],
    [xgb_probs, rf_probs, ann_probs]
):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()

plt.savefig(SAVE_PATH + "roc_curve.png")
plt.close()

# ===============================
# 📊 CONFUSION MATRIX
# ===============================
cm = confusion_matrix(y_test, xgb_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

plt.title("Confusion Matrix - XGBoost")
plt.savefig(SAVE_PATH + "confusion_matrix.png")
plt.close()

# ===============================
# 📊 METRICS GRAPH
# ===============================
metrics_df.set_index("Model").plot(kind="bar")

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)

plt.savefig(SAVE_PATH + "metrics_comparison.png")
plt.close()

# ===============================
# 📉 ANN LOSS CURVE
# ===============================
plt.figure()
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.title("ANN Loss Curve")

plt.savefig(SAVE_PATH + "ann_loss.png")
plt.close()

# ===============================
# 💾 SAVE MODELS
# ===============================
joblib.dump(xgb, SAVE_PATH + "xgb_model.pkl")
joblib.dump(rf, SAVE_PATH + "rf_model.pkl")
ann.save(SAVE_PATH + "ann_model.keras")

joblib.dump(imputer, SAVE_PATH + "imputer.pkl")
joblib.dump(scaler, SAVE_PATH + "scaler.pkl")
joblib.dump(selector, SAVE_PATH + "selector.pkl")

print("✅ EVERYTHING (MODELS + GRAPHS + METRICS) SAVED TO DRIVE!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Shape: (1985, 28)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - accuracy: 0.7055 - loss: 0.5572 - val_accuracy: 0.8113 - val_loss: 0.4244
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8291 - loss: 0.3733 - val_accuracy: 0.8365 - val_loss: 0.3130
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8677 - loss: 0.2962 - val_accuracy: 0.9560 - val_loss: 0.2455
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9213 - loss: 0.2265 - val_accuracy: 0.9591 - val_loss: 0.2032
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9370 - loss: 0.1895 - val_accuracy: 0.9591 - val_loss: 0.1655
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9425 - loss: 0.1612 - val_accuracy: 0.9654 - val_loss: 0.1416
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9646 - loss: 0.1287 - val_accuracy: 0.9623 - val_loss: 0.1218
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9709 - loss: 0.1084 - val_accuracy: 0.9654 - val_loss

In [ ]:
from google.colab import files

SAVE_PATH = "/content/drive/MyDrive/major project code implementation/"

files.download(SAVE_PATH + "xgb_model.pkl")
files.download(SAVE_PATH + "rf_model.pkl")
files.download(SAVE_PATH + "ann_model.keras")
files.download(SAVE_PATH + "imputer.pkl")
files.download(SAVE_PATH + "scaler.pkl")
files.download(SAVE_PATH + "selector.pkl")
files.download(SAVE_PATH + "encoders.pkl")
files.download(SAVE_PATH + "feature_names.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("XGB Mean Prediction:", np.mean(xgb.predict(X_test)))
print("RF Mean Prediction:", np.mean(rf.predict(X_test)))

XGB Mean Prediction: 0.5440806045340051
RF Mean Prediction: 0.5390428211586902


In [4]:
# ===============================
# INSTALL
# ===============================
!pip install -q gradio

# ===============================
# IMPORTS
# ===============================
import gradio as gr
import pandas as pd
import numpy as np
import joblib

# ===============================
# PATH
# ===============================
SAVE_PATH = "/content/drive/MyDrive/major project code implementation/"

# ===============================
# LOAD MODELS
# ===============================
xgb = joblib.load(SAVE_PATH + "xgb_model.pkl")
rf = joblib.load(SAVE_PATH + "rf_model.pkl")

imputer = joblib.load(SAVE_PATH + "imputer.pkl")
scaler = joblib.load(SAVE_PATH + "scaler.pkl")
selector = joblib.load(SAVE_PATH + "selector.pkl")
encoders = joblib.load(SAVE_PATH + "encoders.pkl")

# ✅ TRUE TRAINING FEATURES
feature_names = imputer.feature_names_in_

print("✅ Models Loaded")

# ===============================
# HELPER
# ===============================
def yn_to_num(x):
    return 1 if x == "Yes" else 0

# ===============================
# PREDICT FUNCTION
# ===============================
def predict(age, gender, jaundice, family_history,
            A1,A2,A3,A4,A5,A6,A7,A8,A9,A10):

    try:
        # Convert answers
        answers = [yn_to_num(x) for x in [A1,A2,A3,A4,A5,A6,A7,A8,A9,A10]]
        A_score = sum(answers)

        # ===============================
        # CREATE INPUT STRUCTURE
        # ===============================
        df = pd.DataFrame(np.zeros((1, len(feature_names))), columns=feature_names)

        # Fill main values
        mapping = {
            "Age_Years": age,
            "Gender": gender,
            "Jaundice": jaundice,
            "Family_History": family_history,
        }

        for col, val in mapping.items():
            if col in df.columns:
                df[col] = val

        # Fill A1–A10
        for i in range(10):
            col = f"A{i+1}_Score"
            if col in df.columns:
                df[col] = answers[i]

        # Required column
        if "Who_completed_the_test" in df.columns:
            if "Who_completed_the_test" in encoders:
                df["Who_completed_the_test"] = encoders["Who_completed_the_test"].classes_[0]
            else:
                df["Who_completed_the_test"] = "Parent"

        # ===============================
        # ENCODING
        # ===============================
        for col in df.columns:
            if col in encoders:
                try:
                    df[col] = encoders[col].transform(df[col])
                except:
                    df[col] = 0

        # ===============================
        # PREPROCESS
        # ===============================
        X = imputer.transform(df)
        X = scaler.transform(X)
        X = selector.transform(X)

        # ===============================
        # MODEL PREDICTIONS
        # ===============================
        xgb_prob = float(xgb.predict_proba(X)[0][1])
        rf_prob = float(rf.predict_proba(X)[0][1])
        avg_prob = (xgb_prob + rf_prob) / 2

        # ===============================
        # 🔥 HYBRID LOGIC (KEY FIX)
        # ===============================
        if A_score >= 7 or avg_prob > 0.7:
            result = "🔴 High Risk of ASD"
        elif A_score >= 4 or avg_prob > 0.4:
            result = "🟡 Moderate Risk"
        else:
            result = "🟢 Low Risk"

        return f"""
🧠 ASD Risk Score: {avg_prob:.2f}

{result}

A1–A10 Score: {A_score}/10

--- Model Breakdown ---
XGBoost: {xgb_prob:.2f}
Random Forest: {rf_prob:.2f}
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"

# ===============================
# UI
# ===============================
app = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Age", value=10),
        gr.Dropdown(["Male","Female"], value="Male", label="Gender"),
        gr.Dropdown(["Yes","No"], value="No", label="Jaundice"),
        gr.Dropdown(["Yes","No"], value="No", label="Family History"),

        gr.Radio(["Yes","No"], value="No", label="A1: Social difficulty"),
        gr.Radio(["Yes","No"], value="No", label="A2: Poor eye contact"),
        gr.Radio(["Yes","No"], value="No", label="A3: No response to name"),
        gr.Radio(["Yes","No"], value="No", label="A4: Repetitive behavior"),
        gr.Radio(["Yes","No"], value="No", label="A5: Emotion difficulty"),
        gr.Radio(["Yes","No"], value="No", label="A6: Avoids interaction"),
        gr.Radio(["Yes","No"], value="No", label="A7: Speech delay"),
        gr.Radio(["Yes","No"], value="No", label="A8: Fixated interests"),
        gr.Radio(["Yes","No"], value="No", label="A9: Sensory issues"),
        gr.Radio(["Yes","No"], value="No", label="A10: Trouble with change"),
    ],
    outputs="text",
    title="🧠 ASD Screening Tool (Final Hybrid Model)",
    description="AI + Rule-based system for realistic ASD prediction"
)

# ===============================
# RUN
# ===============================
app.launch(share=True)


# ===============================
# DASHBOARD UI OVERRIDE (PASTE AT END)
# ===============================

def dashboard_predict(age, gender, jaundice, family_history,
                      A1,A2,A3,A4,A5,A6,A7,A8,A9,A10):

    # 👉 call your existing predict function
    output = predict(age, gender, jaundice, family_history,
                     A1,A2,A3,A4,A5,A6,A7,A8,A9,A10)

    # If error, show as is
    if "Error" in output:
        return f"<h3 style='color:red'>{output}</h3>"

    # Extract values from your existing output
    try:
        import re

        avg_prob = float(re.search(r"Score: ([0-9.]+)", output).group(1))
        A_score = int(re.search(r"A1–A10 Score: ([0-9]+)", output).group(1))
        xgb_prob = float(re.search(r"XGB: ([0-9.]+)", output).group(1))
        rf_prob = float(re.search(r"RF: ([0-9.]+)", output).group(1))

    except:
        return f"<pre>{output}</pre>"

    risk_percent = int(avg_prob * 100)

    color = "green"
    if avg_prob > 0.7:
        color = "#e74c3c"
        result = "🔴 High Risk"
    elif avg_prob > 0.4:
        color = "#f39c12"
        result = "🟡 Moderate Risk"
    else:
        result = "🟢 Low Risk"

    return f"""
    <div style="font-family:Segoe UI; background:#f4f6f9; padding:20px; border-radius:15px;">

        <h1 style="text-align:center; color:#2c3e50;">🧠 ASD Dashboard</h1>

        <div style="display:flex; gap:20px;">

            <div style="flex:1; background:white; padding:20px; border-radius:12px;">
                <h3>📊 Risk Overview</h3>
                <h2 style="color:{color};">{risk_percent}%</h2>

                <div style="background:#ddd; border-radius:10px; height:20px;">
                    <div style="width:{risk_percent}%; background:{color}; height:100%; border-radius:10px;"></div>
                </div>

                <p><b>Status:</b> {result}</p>
                <p><b>A Score:</b> {A_score}/10</p>
            </div>

            <div style="flex:1; background:white; padding:20px; border-radius:12px;">
                <h3>🔍 Model Output</h3>
                <p>XGBoost: {xgb_prob:.2f}</p>
                <p>Random Forest: {rf_prob:.2f}</p>
            </div>

        </div>

    </div>
    """


# ===============================
# NEW DASHBOARD UI
# ===============================
import gradio as gr

app = gr.Interface(
    fn=dashboard_predict,   # 👈 using wrapper
    inputs=[
        gr.Number(label="Age", value=10),
        gr.Dropdown(["Male","Female"], value="Male", label="Gender"),
        gr.Dropdown(["Yes","No"], value="No", label="Jaundice"),
        gr.Dropdown(["Yes","No"], value="No", label="Family History"),

        gr.Radio(["Yes","No"], value="No", label="A1"),
        gr.Radio(["Yes","No"], value="No", label="A2"),
        gr.Radio(["Yes","No"], value="No", label="A3"),
        gr.Radio(["Yes","No"], value="No", label="A4"),
        gr.Radio(["Yes","No"], value="No", label="A5"),
        gr.Radio(["Yes","No"], value="No", label="A6"),
        gr.Radio(["Yes","No"], value="No", label="A7"),
        gr.Radio(["Yes","No"], value="No", label="A8"),
        gr.Radio(["Yes","No"], value="No", label="A9"),
        gr.Radio(["Yes","No"], value="No", label="A10"),
    ],
    outputs=gr.HTML(),
    title="🧠 ASD Prediction Dashboard",
    description="Visual dashboard for ASD screening"
)

app.launch(share=True)

✅ Models Loaded
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://060f3fff1a9c6e6d4f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7b06703b7e5be69b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
